In [ ]:
!pip install -q optuna
!pip install -q catboost

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import numpy as np
from catboost import CatBoostClassifier

In [ ]:
import optuna

In [ ]:
# Поместите файлы данных в рабочую папку ноутбука.


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
orders = pd.read_parquet("orders.parquet")
products = pd.read_parquet("products.parquet")

In [ ]:
train.head()

In [ ]:
orders.head()

In [ ]:
products.head()

In [ ]:
for df in [train, test]:
    df["scoring_date"] = pd.to_datetime(df["scoring_date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])

In [ ]:
train.shape, test.shape, orders.shape, products.shape


In [ ]:
orders_full = orders.merge(products, on="product_id", how="left")

In [ ]:
orders_full.head()

EDA

In [ ]:
print("orders:", list(orders.columns))
print("products:", list(products.columns))
print("train:", list(train.columns))

In [ ]:
orders_full.gender_product.unique()

In [ ]:
orders_full.category.unique()

In [ ]:
orders_full.brand.nunique()

In [ ]:
def features(base_df, orders_df):
    keys = ["customer_id", "scoring_date"]
    base_clean = base_df[[col for col in ["customer_id", "scoring_date", "target"]if col in base_df.columns]].copy()
    base_clean["scoring_date"] = pd.to_datetime(base_clean["scoring_date"])
    merged = base_clean[keys].drop_duplicates().merge(orders_df, on="customer_id", how="left")
    merged["order_date"] = pd.to_datetime(merged["order_date"])
    cutoff = merged["scoring_date"] - pd.Timedelta(days=14)
    merged = merged[merged["order_date"] <= cutoff].copy()
    merged["days_since_cutoff"] = merged["scoring_date"]- pd.Timedelta(days=14)- merged["order_date"].dt.days
    merged["order_ym"] = merged["order_date"].dt.to_period("M")
    merged["has_discount"] = (merged["discount_percent"] > 0).astype(int)

    orders_lvl = merged.groupby(keys + ["order_id"], as_index=False).agg(
            order_date=("order_date", "first"),
            days_since_cutoff=("days_since_cutoff", "first"),
            order_ym=("order_ym", "first"),
            order_spent=("paid_price", "sum"),
            order_items=("product_id", "count"),
            order_discount=("discount_percent", "mean"),
            order_loyalty=("loyalty_percent", "mean"),
        ).sort_values(keys + ["order_date", "order_id"])

    orders_lvl["days_diff"] = orders_lvl.groupby(keys)["order_date"].diff().dt.days
    order_feats = (
        orders_lvl.groupby(keys, as_index=False).agg(
            recency_days=("days_since_cutoff", "min"),
            tenure_days=("days_since_cutoff", "max"),
            unique_orders=("order_id", "count"),
            avg_order_value=("order_spent", "mean"),
            avg_basket_size=("order_items", "mean"),
            inter_order_mean=("days_diff", "mean"),
            inter_order_std=("days_diff", "std"),
            last_inter_order=("days_diff", "last"),
            active_months=("order_ym", "nunique"),
            mean_discount=("order_discount", "mean"),
            mean_loyalty=("order_loyalty", "mean"),
        )
    )

    for w in (30, 90, 180):
        window = orders_lvl[orders_lvl["days_since_cutoff"] <= w].groupby(keys, as_index=False).agg(
                **{
                    f"orders_{w}d": ("order_id", "count"),
                    f"spent_{w}d": ("order_spent", "sum"),
                })

        order_feats = order_feats.merge(window, on=keys, how="left")
        order_feats[f"orders_{w}d"] = order_feats[f"orders_{w}d"].fillna(0)
        order_feats[f"spent_{w}d"] = order_feats[f"spent_{w}d"].fillna(0)

    order_feats = order_feats.drop(columns=["spent_30d"])

    order_feats["order_frequency"] = order_feats["unique_orders"] / (order_feats["tenure_days"] + 1.0)
    order_feats["recency_to_inter_mean"] = order_feats["recency_days"] / (order_feats["inter_order_mean"] + 1.0)
    order_feats["expected_next_order_days"] = order_feats["inter_order_mean"] - order_feats["recency_days"]
    order_feats["orders_ratio_90_to_all"] = order_feats["orders_90d"] / (order_feats["unique_orders"] + 1e-5)
    order_feats["orders_ratio_30_to_90"] = order_feats["orders_30d"] / (order_feats["orders_90d"] + 1e-5)
    order_feats["orders_ratio_90_to_180"] = order_feats["orders_90d"] / (order_feats["orders_180d"] + 1e-5)
    previous_90_orders = order_feats["orders_180d"] - order_feats["orders_90d"]
    order_feats["orders_trend_90d"] = (order_feats["orders_90d"] + 1) / (previous_90_orders + 1)
    older_orders = order_feats["unique_orders"] - order_feats["orders_90d"]
    older_spent = order_feats["avg_order_value"] * order_feats["unique_orders"] - order_feats["spent_90d"]
    recent_avg_check = order_feats["spent_90d"] / order_feats["orders_90d"].replace(0, np.nan)
    older_avg_check = older_spent / older_orders.replace(0, np.nan)
    order_feats["avg_check_recent_to_old"] = recent_avg_check / older_avg_check.replace(0, np.nan)
    order_feats["inter_order_cv"] = order_feats["inter_order_std"] / order_feats["inter_order_mean"].replace(0, np.nan)

    last_month = orders_lvl.groupby(keys)["order_ym"].transform("max")

    last_month_counts = orders_lvl[orders_lvl["order_ym"] == last_month].groupby(keys, as_index=False).agg(orders_last_active_month=("order_id", "count"))
    order_feats = order_feats.merge(last_month_counts, on=keys, how="left")
    order_feats["last_active_month_order_share"] = order_feats["orders_last_active_month"] / order_feats["unique_orders"]

    last_order = orders_lvl.drop_duplicates(keys, keep="last")[keys + ["order_spent", "order_items"]]
    last_order = last_order.rename(columns={"order_spent": "last_order_value", "order_items": "last_order_items"})

    order_feats = order_feats.merge(last_order, on=keys, how="left")
    order_feats = order_feats.drop(columns=["tenure_days"])

    item_feats = merged.groupby(keys, as_index=False).agg(
            total_items=("product_id", "count"),
            unique_products=("product_id", "nunique"),
            unique_brands=("brand", "nunique"),
            discount_items_share=("has_discount", "mean"),
        )

    item_feats["repeat_product_ratio"] = 1.0 - item_feats["unique_products"] / (item_feats["total_items"] + 1e-5)
    item_feats = item_feats.drop(columns=["total_items", "unique_products"])

    cat_cols_raw = ["gender_product", "category", "category2"]
    top_dfs = []

    for col in cat_cols_raw:
        top_df = merged.dropna(subset=[col]).groupby(keys + [col], as_index=False).size().sort_values("size", ascending=False).drop_duplicates(keys)
        top_df = top_df.rename(columns={col: f"top_{col}"})[keys + [f"top_{col}"]]

        top_dfs.append(top_df)

    res = base_clean.merge(order_feats, on=keys, how="left").merge(item_feats, on=keys, how="left")

    for top_df in top_dfs:
        res = res.merge(top_df, on=keys, how="left")

    zero_cols = [
        "unique_orders",
        "active_months",
        "orders_30d",
        "orders_90d",
        "spent_90d",
        "orders_180d",
        "spent_180d",
        "order_frequency",
        "orders_ratio_90_to_all",
        "orders_ratio_30_to_90",
        "orders_ratio_90_to_180",
        "unique_brands",
        "orders_last_active_month",
    ]
    res[zero_cols] = res[zero_cols].fillna(0)

    for col in cat_cols_raw:
        name = f"top_{col}"
        res[name] = res[name].fillna("unknown").astype(str)

    return res

In [ ]:
train = features(train, orders_full)
test = features(test, orders_full)

In [ ]:
train.head()

In [ ]:
dates = sorted(train["scoring_date"].unique())
split_date = dates[int(len(dates) * 0.8)]

Train = train[train["scoring_date"] < split_date]
Val = train[train["scoring_date"] >= split_date]

feature_cols = [
    c for c in train.columns
    if c not in ["customer_id", "scoring_date", "target"]
]

X_train = Train[feature_cols].copy()
y_train = Train["target"].copy()

X_val = Val[feature_cols].copy()
y_val = Val["target"].copy()

X_train.shape, X_val.shape

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
def ap_at_20(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n = len(y_true)
    k = int(np.ceil(0.2 * n))
    n_pos = y_true.sum()
    denom = min(k, n_pos)

    if denom == 0:
        return 0.0

    order = np.argsort(-y_score)
    y_sorted = y_true[order][:k]

    precisions = np.cumsum(y_sorted) / np.arange(1, k + 1)
    return (precisions * y_sorted).sum() / denom


def objective_lgb(trial, X_tr, y_tr, X_val, y_val):
    lgb_tr = lgb.Dataset(X_tr, y_tr)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_tr)

    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'seed': 42,
        'verbose': -1,

        'eta': trial.suggest_float('eta', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 80),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.8, 1),
        'max_depth': trial.suggest_int('max_depth', 3, 20)
    }

    model = lgb.train(
        params, lgb_tr,
        num_boost_round=2000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )

    predictions = model.predict(X_val)
    predictions = model.predict(X_val)
    score = ap_at_20(y_val, predictions)

    return score


In [ ]:
for col in X_train.columns:
    if X_train[col].dtype == "object":
      X_train[col] = X_train[col].astype("category")
      X_val[col] = X_val[col].astype("category")
      test[col] = test[col].astype("category")

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(lambda trial: objective_lgb(trial, X_train, y_train, X_val, y_val), n_trials=20)

In [ ]:
test.head()

In [ ]:
best_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "seed": 42,
    "verbose": -1,
    "n_jobs": -1,
    **study.best_params,
}

lgb_tr = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_tr)

val_model = lgb.train(
    best_params,
    lgb_tr,
    num_boost_round=2000,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(200, verbose=False)],
)

best_iter = val_model.best_iteration
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype(X_train[col].dtype)

X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_full, y_full),
    num_boost_round=best_iter,
)
X_test = test[feature_cols]
test = test
test["score"] = final_model.predict(X_test)

sub = pd.read_csv("sample_submission.csv")

sub = sub[["customer_id"]].merge(
    test[["customer_id", "score"]],
    on="customer_id",
    how="left",
    validate="one_to_one"
)
sub.to_csv("submission.csv", index=False)

print(f"Лучший AP@20% на временной валидации: {study.best_value:.5f}")
print(f"Число деревьев: {best_iter}")
print(sub.head())

Это была первая попытка, газ пробую катбустик

In [ ]:
feature_cols = [c for c in train.columns if c not in ["customer_id", "scoring_date", "target", "score"]]
cat_cols = train[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()

X_all = train[feature_cols].copy()
y_all = train["target"]
X_test = test[feature_cols].copy()

for col in cat_cols:
    X_train[col] = X_all[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_test[col] = X_test[col].astype(str)

In [ ]:
def objective_cb(trial):
    params = {
        "iterations": 2000,
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "random_seed": 42,
        "thread_count": -1,
        "verbose": 0,

        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 15.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5.0, log=True),
    }

    model = CatBoostClassifier(**params)
    model.fit(
        X_train, y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=70,
        verbose=False
    )

    preds = model.predict_proba(X_val)[:, 1]
    score = ap_at_20(y_val, preds)

    trial.set_user_attr("best_iteration", model.get_best_iteration())
    return score

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)
study_cb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_cb.optimize(objective_cb, n_trials=10)

In [ ]:
X_full_cb = pd.concat([X_train, X_val], axis=0)
y_full_cb = pd.concat([y_train, y_val], axis=0)
best_iter_cb = study_cb.best_trial.user_attrs["best_iteration"]

cat_cols_cb = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols_cb:
    for df in (X_full_cb, X_test):
        df[col] = df[col].astype("object").fillna("unknown").astype(str)

final_cb = CatBoostClassifier(
    iterations=best_iter_cb,
    loss_function="Logloss",
    random_seed=42,
    thread_count=-1,
    verbose=200,
    **study_cb.best_params,
)

final_cb.fit(X_full_cb, y_full_cb, cat_features=cat_cols_cb)

predictions_cb = final_cb.predict_proba(X_test)[:, 1]
pred_cb = pd.DataFrame({
    "customer_id": test["customer_id"].to_numpy(),
    "score": predictions_cb,
})

sub = pd.read_csv("sample_submission.csv")
sub = sub[["customer_id"]].merge(
    pred_cb,
    on="customer_id",
    how="left",
    validate="one_to_one",
)
sub.to_csv("submission_2.csv", index=False)

print(f"Лучший AP@20%: {study_cb.best_value:.5f}")
print(f"Число деревьев: {best_iter_cb}")
print("Сохранено в submission_2.csv")
print(sub.head())

Методом проб и ошибок получился лучший результат все же на последнем LightGMB)